<a href="https://colab.research.google.com/github/vishvaragavanp2024aiml-lang/241501249-Deep-Learning-Lab/blob/main/EX%2010.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install Required Dependencies

In [1]:
!pip install torchmetrics pytorch-fid

### Step 1 & 2: Imports, Clone StyleGAN2-ADA Repository, and Download Pretrained Model

In [2]:
import os
import sys
import subprocess
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets as dsets, transforms
from torchvision.utils import save_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

REPO = 'stylegan2-ada-pytorch'
REPO_URL = 'https://github.com/NVlabs/stylegan2-ada-pytorch.git'

def clone_repo():
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', REPO_URL])
    if os.path.abspath(REPO) not in sys.path:
        sys.path.append(os.path.abspath(REPO))

MODEL_URL = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl'
MODEL_PATH = 'ffhq.pkl'

def download_model():
    if not os.path.exists(MODEL_PATH):
        import urllib.request
        print("Downloading pretrained StyleGAN2-ADA model...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
        print("Download finished.")

Using device: cpu


### Step 3: Generate Synthetic Images

In [3]:
def generate_images(num_images=5):
    import legacy
    import dnnlib

    with open(MODEL_PATH, 'rb') as f:
        G = legacy.load_network_pkl(f)['G_ema'].to(device)

    os.makedirs('generated_images', exist_ok=True)
    imgs = []
    for i in range(num_images):
        z = torch.randn([1, G.z_dim], device=device)
        label = torch.zeros([1, G.c_dim], device=device)
        img = (G(z, label, truncation_psi=0.5, noise_mode='const') + 1) * 0.5
        save_image(img, f'generated_images/fake_{i}.png')
        imgs.append(img.cpu())
    return torch.cat(imgs, dim=0)

### Step 4 & 5: Calculate Inception Score and Prepare Real Images

In [4]:
from torchmetrics.image.inception import InceptionScore

def calculate_inception_score(imgs):
    # Ensure inputs are scaled to 0-255 uint8 format
    dataset = TensorDataset((imgs * 255).clamp(0, 255).to(torch.uint8))
    loader = DataLoader(dataset, batch_size=32)
    is_metric = InceptionScore(normalize=False).to(device)
    for batch in loader:
        is_metric.update(batch[0].to(device))
    mean, std = is_metric.compute()
    print(f"Inception Score: {mean:.2f} ± {std:.2f}")

def prepare_real_images(num_images=5):
    dataset = dsets.CelebA(root='./data', split='train',
                          transform=transforms.Compose([
                              transforms.Resize(512),
                              transforms.CenterCrop(512),
                              transforms.ToTensor()
                          ]),
                          download=True)
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    os.makedirs('real_images', exist_ok=True)
    for i, (img, _) in enumerate(loader):
        if i >= num_images:
            break
        save_image(img, f"real_images/real_{i}.png")

### Step 6 & 7: Compute FID and Run Pipeline

In [ ]:
def calculate_fid():
    subprocess.run(["python", "-m", "pytorch_fid", "real_images", "generated_images"])

def main():
    clone_repo()
    download_model()
    # Increase counts from 1 to 5 for better metrics estimation
    imgs = generate_images(num_images=5)
    calculate_inception_score(imgs)
    prepare_real_images(num_images=5)
    calculate_fid()

if __name__ == "__main__":
    main()